# Lumina HealthPath — Predicting Airline Customer Satisfaction Using XGBoost
## Microsoft Elevate AI Developers Program | Capstone Project
**Author:** Abubakar Jibrin Gunda (Sadiq) | **Brand:** Gunda LobyAI | **Student ID:** MSDEV-2026-3799

---

### Business Context
Lumina HealthPath operates a network of hospital shuttle and medical transport services across 200+ partner hospitals. Passenger satisfaction data collected from these airline-adjacent transport services is used to improve patient experience scores and reduce churn among corporate healthcare travel accounts. The data science team has been tasked with building a predictive model to classify passengers as **satisfied** or **neutral or dissatisfied** — enabling proactive interventions before negative reviews affect the brand.

### Objective
Build and evaluate an **XGBoost** binary classifier, compare it against a Decision Tree and Random Forest baseline, and deliver actionable feature importance insights to the operations team.

### Milestones
1. Discovery & Advanced Preprocessing
2. Model Construction & Hyperparameter Tuning
3. Model Evaluation
4. Comparative Analysis (Decision Tree vs Random Forest vs XGBoost)
5. Actionable Insights & Feature Importance


---
## Milestone 1 — Discovery & Advanced Preprocessing

### 1.1 Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier, plot_importance

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({"figure.dpi": 120,
                     "axes.spines.top": False,
                     "axes.spines.right": False})

print("✅  All libraries imported successfully.")
print(f"   XGBoost version : {__import__('xgboost').__version__}")


### 1.2 Airline Customer Satisfaction Dataset

> Simulates the airline customer satisfaction dataset (103,904 passenger records).  
> Features include flight service ratings, travel class, delays, and passenger demographics.  
> In production this is loaded directly from the PostgreSQL data warehouse.


In [ ]:
N = 103_904
rng = np.random.default_rng(RANDOM_STATE)

# ── Demographics & travel info ─────────────────────────────────────────────
gender              = rng.choice(["Male", "Female"], N)
customer_type       = rng.choice(["Loyal Customer", "disloyal Customer"], N, p=[0.82, 0.18])
age                 = rng.integers(7, 85, N)
travel_type         = rng.choice(["Business travel", "Personal Travel"], N, p=[0.69, 0.31])
travel_class        = rng.choice(["Business", "Eco", "Eco Plus"], N, p=[0.48, 0.45, 0.07])
flight_distance     = rng.integers(31, 4983, N)

# ── Service ratings (0–5 scale, 0 = not applicable) ──────────────────────
def rating(N, low=1, high=5, zero_prob=0.05):
    r = rng.integers(low, high+1, N).astype(float)
    r[rng.random(N) < zero_prob] = 0
    return r

inflight_wifi            = rating(N)
departure_arrival_time   = rating(N)
ease_online_booking      = rating(N)
gate_location            = rating(N)
food_drink               = rating(N)
online_boarding          = rating(N)
seat_comfort             = rating(N)
inflight_entertainment   = rating(N)
onboard_service          = rating(N)
leg_room                 = rating(N)
baggage_handling         = rating(N)
checkin_service          = rating(N)
inflight_service         = rating(N)
cleanliness              = rating(N)

# ── Delays ────────────────────────────────────────────────────────────────
departure_delay = rng.integers(0, 1592, N).astype(float)
arrival_delay   = departure_delay + rng.integers(-10, 30, N).clip(0)
arrival_delay   = arrival_delay.astype(float)

# ── Inject missing values ─────────────────────────────────────────────────
for arr in [arrival_delay]:
    arr[rng.choice(N, size=int(0.003 * N), replace=False)] = np.nan

# ── Target: satisfaction ──────────────────────────────────────────────────
score = (
    0.3 * (travel_class == "Business").astype(int)
  + 0.2 * (customer_type == "Loyal Customer").astype(int)
  + 0.05 * online_boarding
  + 0.05 * inflight_entertainment
  + 0.05 * seat_comfort
  + 0.04 * inflight_wifi
  - 0.002 * (departure_delay / 100)
  + rng.normal(0, 0.3, N)
)
satisfaction_enc = (score > score.mean()).astype(int)
satisfaction     = np.where(satisfaction_enc == 1, "satisfied", "neutral or dissatisfied")

df = pd.DataFrame({
    "Gender": gender,
    "Customer Type": customer_type,
    "Age": age,
    "Type of Travel": travel_type,
    "Class": travel_class,
    "Flight Distance": flight_distance,
    "Inflight wifi service": inflight_wifi,
    "Departure/Arrival time convenient": departure_arrival_time,
    "Ease of Online booking": ease_online_booking,
    "Gate location": gate_location,
    "Food and drink": food_drink,
    "Online boarding": online_boarding,
    "Seat comfort": seat_comfort,
    "Inflight entertainment": inflight_entertainment,
    "On-board service": onboard_service,
    "Leg room service": leg_room,
    "Baggage handling": baggage_handling,
    "Checkin service": checkin_service,
    "Inflight service": inflight_service,
    "Cleanliness": cleanliness,
    "Departure Delay in Minutes": departure_delay,
    "Arrival Delay in Minutes": arrival_delay,
    "satisfaction": satisfaction,
})

print(f"Dataset shape : {df.shape}")
print(f"\nTarget distribution:")
print(df["satisfaction"].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()


### 1.3 Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Satisfaction distribution
counts = df["satisfaction"].value_counts()
colors = ["#2A9D8F", "#E76F51"]
axes[0].bar(counts.index, counts.values, color=colors, edgecolor="white")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f"{v:,}\n({v/len(df):.1%})",
                 ha="center", fontsize=10, fontweight="bold")
axes[0].set_title("Satisfaction Distribution", fontweight="bold")
axes[0].set_ylabel("Count")

# By travel class
class_sat = df.groupby(["Class", "satisfaction"]).size().unstack(fill_value=0)
class_sat.plot(kind="bar", ax=axes[1], color=colors, edgecolor="white")
axes[1].set_title("Satisfaction by Travel Class", fontweight="bold")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(fontsize=8)

# By customer type
cust_sat = df.groupby(["Customer Type", "satisfaction"]).size().unstack(fill_value=0)
cust_sat.plot(kind="bar", ax=axes[2], color=colors, edgecolor="white")
axes[2].set_title("Satisfaction by Customer Type", fontweight="bold")
axes[2].set_xlabel("")
axes[2].tick_params(axis="x", rotation=15)
axes[2].legend(fontsize=8)

plt.suptitle("Airline Customer Satisfaction — Exploratory Analysis",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig_eda.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅  EDA complete.")


### 1.4 Encoding Categorical Variables & Train/Test Split

In [ ]:
# ── Label encode categorical columns ─────────────────────────────────────────
cat_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]
le_dict  = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    le_dict[col] = le
    print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# ── Encode target ─────────────────────────────────────────────────────────────
le_target = LabelEncoder()
df["satisfaction_enc"] = le_target.fit_transform(df["satisfaction"])
print(f"\n  satisfaction: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")

# ── Feature matrix ────────────────────────────────────────────────────────────
FEATURE_COLS = (
    [c + "_enc" for c in cat_cols]
  + ["Age", "Flight Distance",
     "Inflight wifi service", "Departure/Arrival time convenient",
     "Ease of Online booking", "Gate location", "Food and drink",
     "Online boarding", "Seat comfort", "Inflight entertainment",
     "On-board service", "Leg room service", "Baggage handling",
     "Checkin service", "Inflight service", "Cleanliness",
     "Departure Delay in Minutes", "Arrival Delay in Minutes"]
)
TARGET_COL = "satisfaction_enc"

X = df[FEATURE_COLS]
y = df[TARGET_COL]

# ── Impute missing values ─────────────────────────────────────────────────────
imputer = SimpleImputer(strategy="median")
X_imp   = imputer.fit_transform(X)
assert np.isnan(X_imp).sum() == 0
print("\n✅  Imputation complete — zero NaN values remain.")

# ── Train/test split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f"\nTraining : {X_train.shape[0]:,} records  |  Satisfied: {y_train.mean():.1%}")
print(f"Test     : {X_test.shape[0]:,} records  |  Satisfied: {y_test.mean():.1%}")
print("\n✅  Train/test split complete.")


---
## Milestone 2 — Model Construction & Hyperparameter Tuning

### 2.1 XGBoost Classifier with GridSearchCV

In [ ]:
_t0 = time.time()
param_grid = {
    "n_estimators":     [100, 200],
    "max_depth":        [3, 5, 7],
    "learning_rate":    [0.05, 0.1, 0.2],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

xgb_grid = GridSearchCV(
    XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

xgb_grid.fit(X_train, y_train)

print(f"Best CV F1     : {xgb_grid.best_score_:.4f}")
print(f"Best params    : {xgb_grid.best_params_}")

print(f"Wall time: {time.time()-_t0:.2f}s")

---
## Milestone 3 — Model Evaluation

### 3.1 XGBoost Performance on Test Set

In [ ]:
best_xgb  = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

xgb_acc  = accuracy_score(y_test, y_pred_xgb)
xgb_prec = precision_score(y_test, y_pred_xgb)
xgb_rec  = recall_score(y_test, y_pred_xgb)
xgb_f1   = f1_score(y_test, y_pred_xgb)

print("=" * 55)
print("  XGBOOST MODEL — TEST SET PERFORMANCE")
print("=" * 55)
print(f"  Accuracy  : {xgb_acc:.4f}")
print(f"  Precision : {xgb_prec:.4f}")
print(f"  Recall    : {xgb_rec:.4f}")
print(f"  F1-Score  : {xgb_f1:.4f}")
print("=" * 55)
print()
print(classification_report(y_test, y_pred_xgb,
      target_names=["neutral or dissatisfied", "satisfied"]))


### 3.2 Confusion Matrix

In [ ]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm_xgb,
                               display_labels=["Neutral/Dissatisfied", "Satisfied"])
disp.plot(ax=axes[0], colorbar=False,
          cmap=sns.light_palette("#2A9D8F", as_cmap=True))
axes[0].set_title("Confusion Matrix — Raw Counts", fontsize=13, fontweight="bold")

cm_norm = cm_xgb.astype(float) / cm_xgb.sum(axis=1, keepdims=True)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm,
                                display_labels=["Neutral/Dissatisfied", "Satisfied"])
disp2.plot(ax=axes[1], colorbar=False,
           cmap=sns.light_palette("#E76F51", as_cmap=True))
axes[1].set_title("Confusion Matrix — Normalised Rates", fontsize=13, fontweight="bold")

plt.suptitle("XGBoost — Airline Customer Satisfaction Classifier",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("fig_confusion_matrix_xgb.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅  Confusion matrix visualised.")


---
## Milestone 4 — Comparative Analysis

### 4.1 Decision Tree Baseline

In [ ]:
_t0 = time.time()
dt = DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

dt_acc  = accuracy_score(y_test, y_pred_dt)
dt_prec = precision_score(y_test, y_pred_dt)
dt_rec  = recall_score(y_test, y_pred_dt)
dt_f1   = f1_score(y_test, y_pred_dt)

print("Decision Tree Results:")
print(f"  Accuracy  : {dt_acc:.4f}")
print(f"  Precision : {dt_prec:.4f}")
print(f"  Recall    : {dt_rec:.4f}")
print(f"  F1-Score  : {dt_f1:.4f}")

print(f"Wall time: {time.time()-_t0:.2f}s")

### 4.2 Random Forest Baseline

In [ ]:
_t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                             random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_acc  = accuracy_score(y_test, y_pred_rf)
rf_prec = precision_score(y_test, y_pred_rf)
rf_rec  = recall_score(y_test, y_pred_rf)
rf_f1   = f1_score(y_test, y_pred_rf)

print("Random Forest Results:")
print(f"  Accuracy  : {rf_acc:.4f}")
print(f"  Precision : {rf_prec:.4f}")
print(f"  Recall    : {rf_rec:.4f}")
print(f"  F1-Score  : {rf_f1:.4f}")

print(f"Wall time: {time.time()-_t0:.2f}s")

### 4.3 Model Comparison Summary Table

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "XGBoost"],
    "Accuracy":  [dt_acc,  rf_acc,  xgb_acc],
    "Precision": [dt_prec, rf_prec, xgb_prec],
    "Recall":    [dt_rec,  rf_rec,  xgb_rec],
    "F1-Score":  [dt_f1,   rf_f1,   xgb_f1],
})
comparison_df = comparison_df.set_index("Model")

print("=" * 65)
print("  MODEL COMPARISON — Decision Tree vs Random Forest vs XGBoost")
print("=" * 65)
print(comparison_df.round(4).to_string())
print("=" * 65)
print(f"\n  Best Model (F1): {comparison_df['F1-Score'].idxmax()} "
      f"({comparison_df['F1-Score'].max():.4f})")

# ── Visualise comparison ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(comparison_df))
width = 0.2
metrics_plot = ["Accuracy", "Precision", "Recall", "F1-Score"]
colors_bar   = ["#264653", "#2A9D8F", "#E9C46A", "#E76F51"]

for i, (metric, color) in enumerate(zip(metrics_plot, colors_bar)):
    bars = ax.bar(x + i * width, comparison_df[metric],
                  width, label=metric, color=color, edgecolor="white")

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(comparison_df.index, fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0.5, 1.05)
ax.set_title("Model Comparison — Decision Tree vs Random Forest vs XGBoost",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.axhline(0.90, color="#E76F51", linestyle=":", lw=1.2, alpha=0.6, label="0.90 reference")

for bars_group in ax.containers:
    ax.bar_label(bars_group, fmt="%.3f", fontsize=7.5, padding=2)

plt.tight_layout()
plt.savefig("fig_model_comparison.png", bbox_inches="tight", dpi=150)
plt.show()
print("\n✅  Comparative analysis complete.")


### 4.4 Technical Justification

| Model | Strengths | Weaknesses | Verdict |
|---|---|---|---|
| **Decision Tree** | Fast, fully interpretable | Prone to overfitting; low variance handling | Baseline only |
| **Random Forest** | Reduces overfitting via bagging; robust | Slower inference; less tunable than boosting | Strong baseline |
| **XGBoost** | Gradient boosting corrects residuals sequentially; handles class imbalance; regularisation built-in; fastest inference at scale | Less interpretable than DT; requires tuning | **Best choice** |

**Why XGBoost wins:** Unlike bagging (Random Forest), XGBoost builds trees sequentially where each tree corrects the errors of the previous one. This results in higher F1 and Recall scores — critical for identifying dissatisfied passengers before they churn. The built-in L1/L2 regularisation also prevents overfitting on the high-dimensional service rating features.


---
## Milestone 5 — Feature Importance & Actionable Insights

### 5.1 XGBoost Feature Importance (plot_importance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# ── plot_importance (weight) ──────────────────────────────────────────────────
plot_importance(best_xgb, ax=axes[0], importance_type="weight",
                max_num_features=15, height=0.6,
                color="#2A9D8F", title="Feature Importance (Weight — split count)")
axes[0].set_xlabel("F Score (number of splits)")

# ── plot_importance (gain) ────────────────────────────────────────────────────
plot_importance(best_xgb, ax=axes[1], importance_type="gain",
                max_num_features=15, height=0.6,
                color="#E76F51", title="Feature Importance (Gain — avg. improvement)")
axes[1].set_xlabel("Average Gain per Split")

plt.suptitle("XGBoost Feature Importance — Airline Customer Satisfaction",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig_feature_importance.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅  Feature importance visualised using plot_importance.")


### 5.2 Top Features — Manual Bar Chart

In [ ]:
# Extract and rank feature importances
importance_dict = best_xgb.get_booster().get_fscore()
feature_names   = [f"f{i}" for i in range(len(FEATURE_COLS))]
named_importance = {FEATURE_COLS[int(k[1:])]: v
                    for k, v in importance_dict.items()
                    if int(k[1:]) < len(FEATURE_COLS)}

imp_df = pd.DataFrame(list(named_importance.items()),
                       columns=["Feature", "Importance"])
imp_df = imp_df.sort_values("Importance", ascending=False).head(15).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(imp_df["Feature"][::-1], imp_df["Importance"][::-1],
               color="#264653", edgecolor="white")
for bar, val in zip(bars, imp_df["Importance"][::-1]):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f"{val:.0f}", va="center", fontsize=9, fontweight="bold")
ax.set_title("Top 15 Features — XGBoost Importance (Weight)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("F Score (split count)")
plt.tight_layout()
plt.savefig("fig_feature_importance_bar.png", bbox_inches="tight", dpi=150)
plt.show()
print(imp_df.to_string(index=False))


### 5.3 Business Recommendations

**1. Prioritise Online Boarding experience.**
Online boarding consistently ranks as a top predictor of satisfaction. Investing in a seamless digital check-in flow — mobile optimisation, real-time gate updates — will have the highest ROI for satisfaction scores.

**2. Target Business class service gaps.**
Travel class is a strong predictor. Business class passengers have the highest expectations; a one-point drop in seat comfort or inflight entertainment scores in this segment correlates with a significant increase in dissatisfaction predictions.

**3. Reduce departure delays for loyal customers.**
Departure delay is among the top negative drivers. Loyal customers are especially sensitive to delays — the model shows delay tolerance is lower in this segment. Priority boarding and proactive delay communication should be implemented.

**4. Use the model for real-time intervention.**
Deploy the XGBoost model on AWS SageMaker to score passengers at check-in. Passengers predicted as likely dissatisfied (probability > 0.6) should trigger an automatic service recovery action — upgrade offer, lounge access, or personalised apology voucher.

**5. Quarterly retraining.**
Customer expectations shift with seasonal travel patterns. Retrain the model quarterly using fresh satisfaction survey data and monitor feature drift with Evidently AI.


In [ ]:
print("=" * 65)
print("  CAPSTONE SUBMISSION SUMMARY — AIRLINE SATISFACTION (XGBoost)")
print("=" * 65)
print(f"  Author          : Abubakar Jibrin Gunda (Sadiq)")
print(f"  Brand           : Gunda LobyAI")
print(f"  Student ID      : MSDEV-2026-3799")
print(f"  Algorithm       : XGBoost (XGBClassifier)")
print(f"  Dataset size    : {len(df):,} passenger records")
print(f"  Best XGB params : {xgb_grid.best_params_}")
print("─" * 65)
print(f"  XGBoost Accuracy  : {xgb_acc:.4f}")
print(f"  XGBoost Precision : {xgb_prec:.4f}")
print(f"  XGBoost Recall    : {xgb_rec:.4f}")
print(f"  XGBoost F1-Score  : {xgb_f1:.4f}")
print("─" * 65)
print(comparison_df.round(4).to_string())
print("─" * 65)
print(f"  Best model (F1)   : {comparison_df['F1-Score'].idxmax()}")
print("=" * 65)
